In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

In [ ]:
%%RecordEventWithColumnInfo
# 
# import dias.rewriter
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import os
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import seaborn as sns  # data visualization
import matplotlib.pyplot as plt


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

vlgr_df = pd.read_csv("/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input/animal-crossing-new-horizons-nookplaza-dataset/villagers.csv")
popul_df = pd.read_csv("/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input/acnh-villager-popularity/acnh_villager_data.csv")
factor = 40
popul_df = pd.concat([popul_df]*factor, ignore_index=True)
vlgr_df = pd.concat([vlgr_df]*factor, ignore_index=True)
print(popul_df.info())
vlgr_df.info()

In [ ]:
%%RecordEventWithColumnInfo
### cell 1 ###

vlgr_df.head()

In [ ]:
%%RecordEventWithColumnInfo
### cell 2 ###

popul_df.head()

In [ ]:
%%RecordEventWithColumnInfo
### cell 3 ###

vlgr_df.info()

In [ ]:
%%RecordEventWithColumnInfo
### cell 4 ###

popul_df.info()

In [ ]:
%%RecordEventWithColumnInfo
### cell 5 ###

# There are some missing/non-matching names 
vlgr_df["Name"].isin(popul_df['name']).sum()

In [ ]:
%%RecordEventWithColumnInfo
### cell 6 ###

# vlgr_df does not have these names...
mismatch_names = popul_df["name"][popul_df["name"].isin(vlgr_df["Name"]) == False]
mismatch_names

In [ ]:
%%RecordEventWithColumnInfo
### cell 7 ###

# Data set is small enough to pick out the same names
# Correcting names in popul_df to match vlgr_df
popul_df['name'] = popul_df['name'].replace(['OHare'],"O\'Hare")
popul_df['name'] = popul_df['name'].replace(['Buck(Brows)'],"Buck")
popul_df['name'] = popul_df['name'].replace(['Renee'],"Renée")
popul_df['name'] = popul_df['name'].replace(['WartJr'],"Wart Jr.")
popul_df['name'] = popul_df['name'].replace(['Crackle(Spork)'],"Spork")

In [ ]:
%%RecordEventWithColumnInfo
### cell 8 ###

# Checking if All names match
vlgr_df["Name"].isin(popul_df['name']).sum()

In [ ]:
%%RecordEventWithColumnInfo
### cell 9 ###

# drop villagers that are in popul_df but not in vlgr_df
popul_df = popul_df.drop(popul_df[popul_df["name"].isin(vlgr_df["Name"]) == False].index)

In [ ]:
%%RecordEventWithColumnInfo
### cell 10 ###

# Now that both df have same length, we can set index as names and combine the 2 dfs
popul_df.set_index('name', drop=True, inplace=True)
vlgr_df.set_index('Name', drop=True, inplace=True)

In [ ]:
%%RecordEventWithColumnInfo
### cell 11 ###

combined_df = popul_df.merge(vlgr_df, left_index=True, right_index=True)

In [ ]:
%%RecordEventWithColumnInfo
### cell 12 ###

# drop irrelevent columns
combined_df.drop(columns=['Furniture List', 'Filename', 'Unique Entry ID', "Wallpaper", "Flooring", "Birthday", "Favorite Song"], inplace=True)

In [ ]:
%%RecordEventWithColumnInfo
### cell 13 ###

combined_df.sort_values(['tier', 'rank'], inplace=True)
combined_df['overall_ranking'] = np.arange(1, len(combined_df)+1)
combined_df.insert(2, 'overall_ranking', combined_df.pop('overall_ranking'))

In [ ]:
%%RecordEventWithColumnInfo
### cell 14 ###

overall_mean = combined_df.overall_ranking.mean()
print(f'The overall_mean is {overall_mean}, this would serve as a baseline for to compare against popularity performance of our features.')

In [ ]:
%%RecordEventWithColumnInfo
### cell 15 ###

combined_df.columns

In [ ]:
%%RecordEventWithColumnInfo
### cell 16 ###

combined_df['Gender'].value_counts()

In [ ]:
%%RecordEventWithColumnInfo
### cell 17 ###

combined_df.groupby('tier').Gender.value_counts()

In [ ]:
%%RecordEventWithColumnInfo
### cell 18 ###

pd.pivot_table(combined_df, index = 'tier', values = 'Catchphrase', columns="Gender", aggfunc='count')

In [ ]:
%%RecordEventWithColumnInfo
### cell 19 ###

# creating value counts dataframe for each species type
species_ranking = combined_df.groupby('Species').mean(numeric_only=True)['overall_ranking'].to_frame().reset_index().sort_values('overall_ranking')
species_ranking

In [ ]:
%%RecordEventWithColumnInfo
### cell 20 ###

combined_df.Personality.value_counts()

In [ ]:
%%RecordEventWithColumnInfo
### cell 21 ###

# creating value counts dataframe for each personality type
personality_ranking = combined_df.groupby('Personality').mean(numeric_only=True)['overall_ranking'].to_frame().reset_index().sort_values('overall_ranking')

In [ ]:
%%RecordEventWithColumnInfo
### cell 22 ###

pd.pivot_table(combined_df, index = 'tier', values = 'Catchphrase', columns="Personality", aggfunc='count')

In [ ]:
%%RecordEventWithColumnInfo
### cell 23 ###

# generating value counts dataframe for each style type
style_ranking1 = combined_df.groupby('Style 1').mean(numeric_only=True)['overall_ranking'].to_frame().reset_index().sort_values('overall_ranking')
style_ranking2 = combined_df.groupby('Style 2').mean(numeric_only=True)['overall_ranking'].to_frame().reset_index().sort_values('overall_ranking')

In [ ]:
%%RecordEventWithColumnInfo
### cell 24 ###

# combining the 2 style columns and finding a mean
style_ranking = style_ranking1.copy()
style_series = (style_ranking1['overall_ranking'] + style_ranking2['overall_ranking'])/2
style_ranking["overall_ranking"] = style_series

In [ ]:
%%RecordEventWithColumnInfo
### cell 25 ###

style_ranking

In [ ]:
%%RecordEventWithColumnInfo
### cell 26 ###

pd.pivot_table(combined_df, index = 'tier', values = 'Catchphrase', columns="Style 1", aggfunc='count')

In [ ]:
%%RecordEventWithColumnInfo
### cell 27 ###

pd.pivot_table(combined_df, index = 'tier', values = 'Catchphrase', columns="Style 2", aggfunc='count')